# QuBridge - Reproducibility Notebook

This notebook reproduces all four data artifacts of the paper from the cached CSVs in `../data/`.

| Section | Paper artifact | CSV |
|---|---|---|
| 1 | Table I + Fig 2(b) - pipeline trace | `table1_pipeline_trace.csv` |
| 2 | Table II + Fig 3(b) - L2 filter cascade | `table2_l2_filter.csv` |
| 3 | Fig 4(b) - L3 per-gate pulse-shape | `fig4b_l3_pulse_shape.csv` |
| 4 | Table III + Fig 5 - encoded teleportation | `table3_encoded_teleportation.csv` |

**No live IBM Quantum hardware access required.** All simulation outputs are cached in the CSVs.

Run sequentially (`Cell -> Run All`) or by section.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA = Path('..') / 'data'
assert DATA.is_dir(), f'data/ not found at {DATA.resolve()}'
list(DATA.glob('*.csv'))

## 1. Table I + Fig 2(b) - Pipeline Trace

Three-stage progressive ablation:

- Stage 0 (Baseline): all 184 L2 paths sampled, L3 fixed at All Square
- Stage 1 (+L2): L2 fixed at noise-aware path (66, 67, 74), L3 varied over 4 shape settings
- Stage 2 (+L2+L3): both fixed at per-gate optimum

**Band** = F* - Min F, where F* = 0.985 is the joint pipeline optimum used as a constant upper reference across rows.

In [ ]:
df1 = pd.read_csv(DATA / 'table1_pipeline_trace.csv')
f_star = 0.985  # joint L2+L3 pipeline optimum

g1 = df1.groupby(['stage', 'stage_label']).agg(
    n=('fidelity', lambda s: s.nunique()),
    min_F=('fidelity', 'min'),
    max_F=('fidelity', 'max'),
).reset_index()
g1['band_pp'] = (f_star - g1['min_F']) * 100
g1.round(3)

**Expected values (Table I in paper):**

| Layer | n | Min F | Max F | Band (pp) |
|---|---|---|---|---|
| Baseline | 184 | 0.867 | 0.982 | 11.8 |
| +L2 | 4 | 0.975 | 0.985 | 1.0 |
| +L2+L3 | 1 | 0.985 | 0.985 | 0.0 |

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
x = range(len(g1))
ax.fill_between(x, g1['min_F'], g1['max_F'], color='#94A3B8', alpha=0.3, label='Fidelity band')
ax.plot(x, g1['max_F'], 'o-', color='#16A34A', label='Max F')
ax.plot(x, g1['min_F'], 's-', color='#2563EB', label='Min F')
ax.axhline(f_star, color='#94A3B8', linestyle=':', label=f'Pipeline optimum F={f_star}')
ax.set_xticks(x)
ax.set_xticklabels(g1['stage_label'])
ax.set_ylabel('Fidelity F')
ax.set_title('Pipeline Convergence (Fig 2b)')
ax.legend(loc='lower right', fontsize=8)
ax.grid(axis='y', linestyle=':', alpha=0.5)
fig.tight_layout()
plt.show()

## 2. Table II + Fig 3(b) - L2 Filter Cascade

Cumulative threshold filtering on IBM Torino. L3 fixed at All Square throughout. Best/worst surviving path per filter level.

In [ ]:
df2 = pd.read_csv(DATA / 'table2_l2_filter.csv')
filters = df2['filter_label'].drop_duplicates().tolist()

g2 = df2.groupby('filter_label').agg(
    n_paths=('n_paths', 'first'),
    worst_F=('fidelity', 'min'),
    best_F=('fidelity', 'max'),
).reindex(filters)
g2['band_pp'] = (f_star - g2['worst_F']) * 100
g2.round(3)

**Expected values (Table II in paper):**

| Filter | Paths | Worst F | Best F | Band (pp) |
|---|---|---|---|---|
| Baseline | 184 | 0.867 | 0.976 | 11.8 |
| +RO<5% | 67 | 0.935 | 0.976 | 5.0 |
| 2Q<1% | 63 | 0.952 | 0.976 | 3.3 |
| 2Q<0.5% | 48 | 0.961 | 0.976 | 2.4 |
| 2Q<0.3% | 27 | 0.966 | 0.976 | **1.9** |

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
x = range(len(g2))
ax.fill_between(x, g2['worst_F'], g2['best_F'], color='#94A3B8', alpha=0.3,
                label='Fidelity band (L3=All Sq.)')
ax.plot(x, g2['best_F'], 'o-', color='#16A34A', label='Best F')
ax.plot(x, g2['worst_F'], 's-', color='#2563EB', label='Worst F')
ax.axhline(f_star, color='#94A3B8', linestyle=':',
           label=f'Pipeline optimum F={f_star}')
ax.set_xticks(x)
ax.set_xticklabels(g2.index, rotation=20, ha='right', fontsize=8)
ax.set_ylabel('Fidelity F')
ax.set_title('L2 Cumulative Filtering (Fig 3b)')
ax.legend(loc='lower right', fontsize=8)
ax.grid(axis='y', linestyle=':', alpha=0.5)
fig.tight_layout()
plt.show()

## 3. Fig 4(b) - L3 Per-Gate Pulse-Shape Comparison

Compare three uniform proxy settings (All Square / All Gaussian Sq. / All DRAG) against the per-gate optimal proxy (SX=DRAG, CZ=Gaussian Sq., Meas=Square).

Data source: Stage 1 of the same `table1_pipeline_trace.csv` (L2 fixed at noise-aware path, L3 swept across 4 shape settings).

In [ ]:
# Stage 1 of e1h has L3 swept across 4 settings with L2 fixed.
df4 = df1[df1['stage'] == 1].copy()
df4.head()

In [ ]:
g4 = df4.groupby('pulse_config')['fidelity'].mean().sort_values()
g4_pct = g4 * 100
print(g4_pct.round(2).to_string())

**Expected (Fig 4(b) in paper):** per-gate optimal achieves F=98.49% (+0.87% over All Square baseline).

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3.5))
colors = ['#E24B4A' if 'Square' in c and 'Gaussian' not in c else
          '#3FA7E6' if 'Gaussian' in c else
          '#EF9F27' if 'DRAG' in c.upper() else '#1D9E75' for c in g4.index]
ax.bar(range(len(g4)), g4_pct, color=colors)
ax.set_xticks(range(len(g4)))
ax.set_xticklabels(g4.index, rotation=15, ha='right', fontsize=8)
ax.set_ylabel('Fidelity (%)')
ax.set_title('L3 Per-gate Pulse-Shape Proxy (Fig 4b)')
ax.set_ylim(g4_pct.min() - 0.5, g4_pct.max() + 0.5)
for i, v in enumerate(g4_pct):
    ax.text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=8)
fig.tight_layout()
plt.show()

## 4. Table III + Fig 5 - Bit-Flip/Parity-Detected Teleportation

Compare physical (3-qubit) vs [[2,1,1]]-encoded (6-qubit) teleportation across noise scales for two input states (|+>, |1>).

- |+> is phase-sensitive - Z error not detected by the code
- |1> is T1-decay-sensitive - bit-flip parity changes detected by syndrome filter

In [ ]:
df3 = pd.read_csv(DATA / 'table3_encoded_teleportation.csv')
df3.columns.tolist()

In [ ]:
# Pivot by (state_label, circuit_type); acceptance = 1 - error_detection_rate.
df3['acceptance'] = 1 - df3['error_detection_rate']
table3 = df3.pivot_table(
    index='noise_scale',
    columns=['state_label', 'circuit_type'],
    values=['fidelity', 'acceptance'],
    aggfunc='first',
)
table3.round(4)

**Expected (Table III in paper):** at noise=1.0, |+> Phys.=0.9849, Log.=0.9769, Accept=0.9263; |1> Phys.=0.9844, Log.=0.9941, Accept=0.9244.

In [ ]:
# Fig 5: physical vs logical (conditional) fidelity per state vs noise.
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True)
for ax, state, title in zip(axes, ['|+>', '|1>'],
                            ['|+> (Z-error dominant)', '|1> (T1 decay)']):
    sub = df3[df3['state_label'] == state].drop_duplicates(['noise_scale', 'circuit_type'])
    phys = sub[sub['circuit_type'] == 'physical'].sort_values('noise_scale')
    log = sub[sub['circuit_type'] == 'logical'].sort_values('noise_scale')
    ax.plot(phys['noise_scale'], phys['fidelity'], 'o-', label='Physical', color='#2563EB')
    ax.plot(log['noise_scale'], log['fidelity'], 's-', label='Logical (cond.)', color='#16A34A')
    ax.set_xlabel('Noise scale')
    ax.set_title(title)
    ax.grid(linestyle=':', alpha=0.5)
    ax.legend(fontsize=8)
axes[0].set_ylabel('Fidelity F')
fig.suptitle('Bit-Flip/Parity-Detected Teleportation (Fig 5)')
fig.tight_layout()
plt.show()

## Done

All four paper artifacts reproduced from cached simulation CSVs. See `../README.md` for the data <-> paper mapping and simulation environment.
